# Базовые операции с геометрией

Геообработка — это набор инструментов для анализа и преобразования пространственных данных. Эти инструменты позволяют:

- выполнять операции с геометрией объектов;
- анализировать взаимное расположение объектов;
- создавать новые наборы данных на основе существующих.

В этом разделе мы рассмотрим наиболее часто используемые базовые инструменты:

- **Буферизация (Buffer)** — создание зон вокруг объектов на заданном расстоянии;
- **Слияние (Dissolve)** — объединение геометрий в одну, целиком или по значению атрибута;
- **Обрезка (Clip)** — отсечение одного набора данных по границам другого;
- **Наложение (Overlay)** — пересечение, объединение и разность двух полигональных слоёв.

Все операции разберём на одной задаче: оценим, какая часть Центрального района Санкт-Петербурга находится в пределах 500 м от остановок общественного транспорта.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import osmnx as ox
import geopandas as gpd

### 0.2 Подготовка данных

Загрузим из OpenStreetMap границу Центрального района Санкт-Петербурга:


In [ ]:
area_name = "Центральный район, Санкт-Петербург"
admin_border = ox.geocode_to_gdf(area_name)

In [ ]:
# Альтернатива: загрузка из сохранённого файла (если osmnx недоступен)
# Скачать файл osm_data.gpkg: https://drive.google.com/file/d/1uhnwDvDwZHhUUum04cp7okzWcvVoQ43Y/view?usp=sharing

# admin_border = gpd.read_file("../data/osm_data.gpkg", layer="central_district")

Также загрузим автобусные остановки и входы в метро:


In [ ]:
tags = {
    "highway": "bus_stop",
    "railway": ["subway_entrance"]
}

stops = ox.features_from_place(area_name, tags)

stops.explore()

In [ ]:
# Альтернатива: загрузка из сохранённого файла (если osmnx недоступен)
# Скачать файл osm_data.gpkg: https://drive.google.com/file/d/1uhnwDvDwZHhUUum04cp7okzWcvVoQ43Y/view?usp=sharing

# stops = gpd.read_file("../data/osm_data.gpkg", layer="transit_stops")

Данные из OSM содержат десятки полей. Оставим только нужные, а идентификатор объекта OSM, который хранится в индексе, перенесём в отдельное поле:


In [ ]:
stops["osm_id"] = stops.index.get_level_values("id")
stops = stops[["osm_id", "railway", "name", "highway", "geometry"]]

stops.head()

## 1. Буферизация (Buffer)

**Буфер** — это область вокруг объекта, все точки которой находятся от него не дальше заданного расстояния. С помощью буферов можно, например, выделить территорию в пределах заданного радиуса от объекта или найти объекты, расположенные поблизости.

Буфер создаётся методом `.buffer()`. Расстояние задаётся **в единицах системы координат**.


### 1.1. Перепроецирование

Наши данные находятся в географической системе координат EPSG:4326, где единицы измерения — градусы. Чтобы построить буфер радиусом 500 **метров**, перепроецируем данные в подходящую зону UTM (подробнее — в разделе [Перепроецирование данных](../module_2/projections_2.ipynb)).

Сразу перепроецируем и границу района — она понадобится нам позже.


In [ ]:
utm_crs = stops.estimate_utm_crs()

stops_utm = stops.to_crs(utm_crs)
admin_border_utm = admin_border.to_crs(utm_crs)

stops_utm.crs

### 1.2. Создание буфера

Построим буферы радиусом 500 м вокруг каждой остановки:


In [ ]:
stops_buffer = stops_utm.copy()
stops_buffer["geometry"] = stops_buffer.geometry.buffer(500)

stops_buffer.explore()

Буферы соседних остановок сильно перекрываются. Чтобы получить единую зону, их нужно слить.

> Буфер показывает расстояние **по прямой**. Реальная пешеходная доступность зависит от уличной сети, и её мы научимся оценивать в модуле «Основы сетевого анализа».


## 2. Слияние (Dissolve)

**Слияние** объединяет несколько геометрий в одну. В GeoPandas оно выполняется методом `.dissolve()`:

- без параметров — все геометрии сливаются в один объект;
- с параметром `by` — геометрии сливаются в группы по значению атрибута (как `groupby` в pandas).

Сольём все буферы в одну зону:


In [ ]:
stops_dissolved_all = stops_buffer.dissolve()

stops_dissolved_all.explore()

Теперь построим отдельные зоны для остановок наземного транспорта и для метро. В поле `railway` у входов в метро указано значение `subway_entrance`, а у остальных объектов — пропуск (`NaN`). Группировать по такому полю неудобно, поэтому создадим поле с типом транспорта:


In [ ]:
stops_buffer.head()

In [ ]:
stops_buffer["transport_type"] = "bus"
stops_buffer.loc[stops_buffer["railway"] == "subway_entrance", "transport_type"] = "subway"

stops_dissolved_cat = stops_buffer.dissolve(by="transport_type").reset_index()

stops_dissolved_cat.explore(column="transport_type")

Мы получили две зоны — для наземного транспорта и для метро. Обе выходят за пределы района.


## 3. Обрезка (Clip)

**Обрезка** оставляет только ту часть объектов, которая попадает внутрь заданной области. Это полезно, когда результаты анализа нужно ограничить территорией исследования.

Оба слоя должны быть в одной системе координат — границу района мы уже перепроецировали.


In [ ]:
stops_clipped = stops_dissolved_cat.clip(admin_border_utm)

stops_clipped.explore(column="transport_type")

Теперь зоны ограничены границей района.


## 4. Наложение (Overlay)

**Наложение** выполняет операции над двумя полигональными слоями так же, как операции над множествами:

| `how=`                   | Результат                                   |
| ------------------------ | ------------------------------------------- |
| `"intersection"`         | только общие части двух слоёв               |
| `"union"`                | все части обоих слоёв                       |
| `"difference"`           | части первого слоя, не покрытые вторым      |
| `"symmetric_difference"` | части, принадлежащие только одному из слоёв |

Чем наложение отличается от обрезки? `clip()` просто обрезает один слой по другому, а `overlay()` **сохраняет атрибуты обоих слоёв** и позволяет выполнять разные операции, например найти разность.


### 4.1. Пересечение (intersection)

Найдём, какая доля территории района находится в пределах 500 м от остановок каждого типа транспорта:


In [ ]:
covered = gpd.overlay(
    admin_border_utm[["geometry"]],
    stops_dissolved_cat[["transport_type", "geometry"]],
    how="intersection"
)
covered


In [ ]:
covered = gpd.overlay(
    admin_border_utm[["geometry"]],
    stops_dissolved_cat[["transport_type", "geometry"]],
    how="intersection"
)

district_area = admin_border_utm.geometry.area.iloc[0]
covered["share_pct"] = covered.geometry.area / district_area * 100

covered[["transport_type", "share_pct"]]

Большая часть района находится в 500-метровой зоне остановок наземного транспорта. Зона метро покрывает заметно меньшую часть территории.


### 4.2. Разность (difference)

Найдём территории района, которые находятся **дальше 500 м от любой остановки**:


In [ ]:
not_covered = gpd.overlay(
    admin_border_utm[["geometry"]],
    stops_dissolved_all[["geometry"]],
    how="difference"
)

share_not_covered = not_covered.geometry.area.sum() / district_area * 100
print(f"Доля территории района дальше 500 м от остановок: {share_not_covered:.1f}%")

m = admin_border_utm.explore(color="gray", style_kwds={"fillOpacity": 0.05})
not_covered.explore(m=m, color="red")

Такие территории — кандидаты на «пробелы» в обслуживании общественным транспортом. Но прежде чем делать выводы, посмотрите на карту: часть из них может приходиться на воду, парки или промышленные территории.


## Итог

В этом разделе мы познакомились с базовыми инструментами работы с геометрией:

- `.buffer()` — построение зон на заданном расстоянии от объектов (только в метрической системе координат);
- `.dissolve()` — слияние геометрий, целиком или по значению атрибута;
- `.clip()` — обрезка слоя по границам территории исследования;
- `gpd.overlay()` — пересечение, объединение и разность полигональных слоёв с сохранением атрибутов обоих.

С их помощью мы оценили, какая часть района находится в пределах 500 м от остановок общественного транспорта.
